[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

/home/user/miniconda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
help(torch.triu)

Help on built-in function triu in module torch:

triu(...)
    triu(input, diagonal=0, *, out=None) -> Tensor

    Returns the upper triangular part of a matrix (2-D tensor) or batch of matrices
    :attr:`input`, the other elements of the result tensor :attr:`out` are set to 0.

    The upper triangular part of the matrix is defined as the elements on and
    above the diagonal.

    The argument :attr:`diagonal` controls which diagonal to consider. If
    :attr:`diagonal` = 0, all elements on and above the main diagonal are
    retained. A positive value excludes just as many diagonals above the main
    diagonal, and similarly a negative value includes just as many diagonals below
    the main diagonal. The main diagonal are the set of indices
    :math:`\lbrace (i, i) \rbrace` for :math:`i \in [0, \min\{d_{1}, d_{2}\} - 1]` where
    :math:`d_{1}, d_{2}` are the dimensions of the matrix.

    Args:
        input (Tensor): the input tensor.
        diagonal (int, optional): the diag

In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    B, seq_q, d_k = Q.shape
    seq_k = K.size(1)

    score = Q @ K.transpose(1, 2) / math.sqrt(d_k)
    score.masked_fill_(torch.triu(torch.ones_like(score), diagonal=1).bool(), float("-inf"))
    weight = torch.softmax(score, dim=-1)
    return weight @ V

In [5]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [6]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (11.0ms)
  ✅ [2/4] Future tokens don't affect past (11.2ms)
  ✅ [3/4] First position only sees itself (1.2ms)
  ✅ [4/4] Gradient flow (18.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (42.1ms total)
  Progress saved. Run status() to see your dashboard.

